# Practice Lab: Computer Use Agents (Lesson 11.6)

Module 11 · 8 exercises · Playwright + CUA + DOM distillation + hybrid perception + sandbox + cost + HITL + India

Exercises 1-4 run locally (pure Python simulations).


# Lesson 11.6: Computer Use Agents

8 exercises: 2E + 3M + 3C

Exercises 1-4 run **locally** (pure Python simulations). Ex 5-8 are design.


In [ ]:
import json
from datetime import datetime


---
## Exercise 1 (Easy): Playwright Browser Agent


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

class SimBrowser:
    ALLOWED=["netsetos.com","wikipedia.org","example.com"]
    MAX=10
    def __init__(self): self.url="about:blank";self.content="";self.history=[];self.steps=0
    def is_safe(self,url): return any(d in url for d in self.ALLOWED)
    def goto(self,url):
        if not self.is_safe(url): self.history.append(f"BLOCKED:{url}"); return False
        self.url=url; self.content={"netsetos.com":"GenAI:14999 INR"}.get(url.replace("https://",""),"Loaded")
        self.history.append(f"goto {url}"); return True
    def decide(self,task):
        if "price" in task.lower() and self.url=="about:blank": return {"action":"goto","url":"https://netsetos.com"}
        if "evil" in task.lower(): return {"action":"goto","url":"https://evil-site.com"}
        return {"action":"done"}
    def run(self,task):
        print(f"  Task: '{task}'")
        for s in range(self.MAX):
            self.steps=s+1; a=self.decide(task)
            if a["action"]=="done": print(f"  Step {self.steps}: DONE"); return {"steps":self.steps,"url":self.url}
            elif a["action"]=="goto":
                ok=self.goto(a["url"]); print(f"  Step {self.steps}: goto {a['url']} [{'OK' if ok else 'BLOCKED'}]")
                if not ok: return {"steps":self.steps,"blocked":a["url"]}
        return {"steps":self.MAX}

print("Playwright Browser Agent:")
print("="*55)
print("\n  Safe URL:"); SimBrowser().run("Find GenAI price on netsetos.com")
print("\n  Blocked URL:"); SimBrowser().run("Go to evil site")
print(f"\n  Safety: allowlist={SimBrowser.ALLOWED}, max={SimBrowser.MAX}")


</details>


---
## Exercise 2 (Easy): Anthropic CUA Screenshot Loop


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class SimCUA:
    def __init__(self,nw=1920,nh=1080,sw=1024,sh=768):
        self.sx=nw/sw;self.sy=nh/sh;self.tokens=0;self.actions=[]
    def scale(self,x,y): return (int(x*self.sx),int(y*self.sy))
    def step(self,task,n):
        tok=1600+735+480+50; self.tokens+=tok
        acts=[{"action":"click","coord":[512,50]},{"action":"type","text":"GenAI"},
              {"action":"key","key":"Enter"},{"action":"scroll","dir":"down","amt":3},
              {"action":"screenshot"}]
        a=acts[min(n-1,len(acts)-1)]; self.actions.append(a); return a,tok

print("Anthropic CUA Loop:")
cua=SimCUA()
print(f"  Resolution: 1920x1080 -> 1024x768 (scale {cua.sx:.2f}x, {cua.sy:.2f}y)")
print(f"\n  Loop (5 steps):")
for s in range(1,6):
    a,t=cua.step("search GenAI",s)
    detail=""
    if "coord" in a:
        cx,cy=a["coord"]; nx,ny=cua.scale(cx,cy)
        detail=f" ({cx},{cy})->native({nx},{ny})"
    elif "text" in a: detail=f": {a['text']}"
    elif "key" in a: detail=f": {a['key']}"
    elif "dir" in a: detail=f": {a['dir']} x{a.get('amt',1)}"
    print(f"  Step {s}: {a['action']}{detail} [{t} tok]")
print(f"\n  Total: {cua.tokens:,} tokens (${cua.tokens*3/1e6:.4f})")
print(f"  5 actions: click, type, key, scroll, screenshot")


</details>


---
## Exercise 3 (Medium): DOM Distillation vs Raw HTML


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
def est_tok(t): return int(len(t.split())*1.3)

raw_html = "<html><head><title>Netsetos</title><link rel=stylesheet href=s.css><script src=a.js></script><style>nav{display:flex}hero{bg:grad}card{br:12px}</style></head><body><nav><a href=/>Home</a><a href=/courses>Courses</a></nav><div class=hero><h1>Learn GenAI</h1><button id=enroll>Enroll</button></div><div class=card><h3>GenAI</h3><p>14999 INR</p><button class=buy data-c=genai>Buy</button></div><div class=card><h3>Python</h3><p>9999 INR</p><button class=buy data-c=python>Buy</button></div><footer>2026</footer><script>analytics tracking animations lazy loading sw</script></body></html>"

distilled = "[1] nav: Home | Courses\n[2] h1: Learn GenAI\n[3] button#enroll: Enroll\n[4] card: GenAI | 14999 INR\n[5] button.buy[genai]: Buy\n[6] card: Python | 9999 INR\n[7] button.buy[python]: Buy"

rt=est_tok(raw_html); dt=est_tok(distilled)
print("DOM Distillation vs Raw HTML:")
print(f"  Raw: {rt} tokens | Distilled: {dt} tokens | Savings: {(1-dt/rt)*100:.0f}% | Ratio: {rt//max(dt,1)}x")
print(f"\n  Distilled DOM:")
for l in distilled.split("\n"): print(f"    {l}")
print(f"\n  LLM: 'click [5]' -> Buy GenAI")
print(f"  1000 pages: raw ${rt*1000*3/1e6:.2f} vs dist ${dt*1000*3/1e6:.2f}")
print(f"  browser-use: 89.1% WebVoyager. Key: give LLM only what it NEEDS")


</details>


---
## Exercise 4 (Medium): Hybrid Perception


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class Hybrid:
    def __init__(self): self.tokens=0;self.modes={"a11y":0,"vision":0}
    def perceive(self,page_type):
        good=["form","table","nav","article","search"]
        if page_type in good:
            self.tokens+=120;self.modes["a11y"]+=1; return {"mode":"a11y","tok":120}
        self.tokens+=1600;self.modes["vision"]+=1; return {"mode":"vision","tok":1600}

h=Hybrid()
pages=[("Standard form","form"),("Data table","table"),("Navigation","nav"),
       ("Canvas app","canvas_app"),("Game UI","game"),("Govt portal","govt_portal"),
       ("Article","article"),("PDF viewer","pdf_viewer")]

print("Hybrid Perception:")
print(f"  {'Page':<20} {'Mode':>8} {'Tokens':>8}")
for name,pt in pages:
    r=h.perceive(pt); print(f"  {name:<20} {r['mode']:>8} {r['tok']:>8}")

pure=len(pages)*1600
print(f"\n  Hybrid: {h.tokens} tok | Pure screenshot: {pure} tok | Savings: {(1-h.tokens/pure)*100:.0f}%")
print(f"  A11y: {h.modes['a11y']}x | Vision fallback: {h.modes['vision']}x")
print(f"  browser-use hybrid: 89.1% vs a11y-only: 73.1%")


</details>


---
## Exercise 5 (Medium): Docker Sandbox
Design/architecture. See practice-lab-11_6.html.


In [ ]:
# Exercise 5: Docker Sandbox
pass


<details><summary>Solution</summary>


In [ ]:
print('Docker Sandbox: Host -> Docker -> Xvfb:99 -> Mutter -> Chromium -> VNC:5900 -> noVNC:6080')
print('Credential proxy on HOST (never in container)')
print('Restrictions: URL allowlist, step=25, timeout=20s/action, 20min/task, FS readonly')
print('Privacy: local OCR masks PII before cloud upload')


</details>


---
## Exercise 6 (Challenge): Cost-Optimized Agent
Design/architecture. See practice-lab-11_6.html.


In [ ]:
# Exercise 6: Cost-Optimized Agent
pass


<details><summary>Solution</summary>


In [ ]:
steps=20;base=(1600+735+480)*steps*3/1e6+50*steps*15/1e6
cache_save=(735+480)*steps*3*0.9/1e6
a11y_save=(1600-120)*steps*0.6*3/1e6
opt=base-cache_save-a11y_save
print(f'Cost-Optimized CUA: baseline ${base:.4f} -> optimized ${max(opt,0.001):.4f}')
print(f'Savings: {(1-max(opt,0.001)/base)*100:.0f}%')
print('Prompt caching 90% + model routing 80% Flash + a11y-first 60%')


</details>


---
## Exercise 7 (Challenge): HITL Approval Flow
Design/architecture. See practice-lab-11_6.html.


In [ ]:
# Exercise 7: HITL Approval Flow
pass


<details><summary>Solution</summary>


In [ ]:
print('HITL Tiers: Auto(85%):navigate,scroll | Supervised(10%):forms | Approval(5%):purchase,delete')
print('TOCTOU Fix: fresh screenshot before critical click, compare expected vs actual')
print('If mismatch: ABORT and re-plan. Use a11y IDs not pixel coords')


</details>


---
## Exercise 8 (Challenge): Indian Portal Automation
Design/architecture. See practice-lab-11_6.html.


In [ ]:
# Exercise 8: Indian Portal Automation
pass


<details><summary>Solution</summary>


In [ ]:
from datetime import datetime
print('Indian Portal: GST pipeline (7 steps), CAPTCHA: EasyOCR 90%+')
print(f'Audit: timestamp={datetime.now().isoformat()}, portal=gst.gov.in')
print('Decision: API exists? Use it. No API? CUA+sandbox+audit')
print('Legal: IT Act 43/66, DPDP consent, IRCTC bots prohibited')


</details>
